# `utils_MemberC.ipynb` — TF-IDF cosine similarity from scratch

**Owner: Member C.**  This notebook documents section *4. MEMBER C — TF-IDF cosine similarity from scratch* of `utils.py`.

The class `TfidfFromScratch` mirrors the standard formulation seen in lecture 3:

* TF (term frequency) — optionally sublinear ($1 + \log \mathrm{tf}$)
* IDF (inverse document frequency) with sklearn's *smooth* variant: $\mathrm{idf}(t) = \log\frac{1 + N}{1 + \mathrm{df}(t)} + 1$
* TF-IDF = TF × IDF
* Final L2 normalisation, so the cosine similarity reduces to a plain dot product.

In [ ]:
import math
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import utils

## A toy corpus to walk through the maths

In [ ]:
docs = [
    'the cat sat on the mat',
    'the dog ran fast',
    'cats and dogs are friends',
    'machine learning is fun',
    'deep learning is a subset of machine learning',
]
for i, d in enumerate(docs):
    print(i, d)

## `fit(documents)`

Two passes over the corpus:

1. **Document-frequency counter.**  For each document we deduplicate the tokens (`set(tokenize(...))`) and increment the document-frequency `df[term]` of every token.
2. **Vocabulary + IDF.**  We keep terms whose `df ≥ min_df` and assign each a column index.  IDF is computed with the smooth formula above.

In [ ]:
tfidf = utils.TfidfFromScratch(sublinear_tf=True, min_df=1).fit(docs)
print('vocabulary size :', len(tfidf.vocabulary_))
print('first 10 tokens :', list(tfidf.vocabulary_)[:10])
print('first 5 idf     :', tfidf.idf_[:5])

## Verifying IDF by hand

In [ ]:
term = 'the'
df_term = sum(1 for d in docs if term in utils.tokenize(d))
expected = math.log((1 + len(docs)) / (1 + df_term)) + 1.0
got = tfidf.idf_[tfidf.vocabulary_[term]]
print(f'df({term}) = {df_term}, expected idf = {expected:.4f}, got = {got:.4f}')

## `transform(documents)` → L2-normalised TF-IDF matrix

For each document we:

1. Compute term counts.
2. Apply `tf = 1 + log(count)` if `sublinear_tf=True`, else raw count.
3. Multiply by `idf[term]`.
4. L2-normalise.

After step 4 every document lies on the unit hypersphere, exactly as covered in the lecture slide *"All documents lie on the unit hypersphere"*.

In [ ]:
M = tfidf.transform(docs)
print('matrix shape :', M.shape)
print('row norms    :', np.linalg.norm(M, axis=1).round(6))   # should all be ≈ 1.0

## `cosine(text1, text2)`

Because rows are unit-norm, $\cos(\theta) = p^\top q$ — i.e. cosine similarity is just a dot product.  This is the feature we expose to the final classifier.

In [ ]:
for i in range(len(docs)):
    for j in range(i+1, len(docs)):
        print(f'cos(d{i}, d{j}) = {tfidf.cosine(docs[i], docs[j]):.4f}')

Notice that documents 3 and 4 (`machine learning is fun` / `deep learning is a subset of machine learning`) have the highest similarity — exactly what a paraphrase detector should reward.

## Cross-check against scikit-learn

The two implementations don't have to be bit-identical (sklearn uses a regex tokeniser; ours uses our `tokenize` helper, which lower-cases and strips punctuation differently), but they should give very close cosine values on this clean corpus.

In [ ]:
sk = TfidfVectorizer(sublinear_tf=True, min_df=1, lowercase=True, token_pattern=r'\S+')
sk.fit(docs)
Msk = sk.transform(docs).toarray()

import pandas as pd
rows = []
for i in range(len(docs)):
    for j in range(i+1, len(docs)):
        ours    = tfidf.cosine(docs[i], docs[j])
        skvalue = float((Msk[i] @ Msk[j]) / (np.linalg.norm(Msk[i]) * np.linalg.norm(Msk[j])))
        rows.append({'pair': f'({i},{j})', 'ours': round(ours, 4), 'sklearn': round(skvalue, 4),
                     'abs_diff': round(abs(ours - skvalue), 4)})
pd.DataFrame(rows)

## Why bother with a from-scratch implementation?

* **Pedagogical.**  We can step through TF, IDF, multiplication and L2 normalisation explicitly, exactly as in the lecture.
* **Sanity check.**  We feed the model *both* our cosine and sklearn's cosine as features (`tfidf_cosine_scratch` and `tfidf_cosine_sklearn`); if XGBoost ends up giving wildly different importances to them, that would tell us something is wrong with one of the two implementations.
* **Customisability.**  Owning the code makes it trivial to swap the tokeniser, change the smoothing constant, or plug in stop-word filtering.